# Exercise 2.2.4: Transforming Data & Creating New Features
*Exercício 2.2.4: Transformar Dados e Criar Novas Variáveis*

This exercise reads the cleaned file produced by **Exercise 2.2.3** (`data/10_cleaned/datania_households_clean.csv`) and turns it into an **analysis-ready** dataset by creating derived columns.

*Este exercício lê o ficheiro limpo produzido pelo **Exercício 2.2.3** (`data/10_cleaned/datania_households_clean.csv`) e transforma-o num conjunto de dados **pronto para análise**, criando colunas derivadas.*

You will practice:
- **Recoding**: per-capita values, age bands with `pd.cut()`, code->label maps with `map()`
- **Conditional assignment** with `np.where()` and `.loc[]`
- Clean, chainable column creation with `assign()` (including dependent columns via lambdas)
- Multi-way categories with `np.select()`
- Custom logic with `apply()`: named functions, lambdas, and row-wise `apply(axis=1)`
- Saving the feature table to `20_processed/`

*Vai praticar:*
- ***Recodificação**: valores per capita, escalões etários com `pd.cut()`, mapas código->rótulo com `map()`*
- ***Atribuição condicional** com `np.where()` e `.loc[]`*
- *Criação de colunas limpa e encadeável com `assign()` (incluindo colunas dependentes através de lambdas)*
- *Categorias com várias saídas usando `np.select()`*
- *Lógica personalizada com `apply()`: funções com nome, lambdas e `apply(axis=1)` linha a linha*
- *Guardar a tabela de variáveis em `20_processed/`*

> **Pipeline:** run Exercise 2.2.3 first so the cleaned file exists. This notebook writes to `20_processed/`.

> ***Fluxo de trabalho:** execute primeiro o Exercício 2.2.3 para que o ficheiro limpo exista. Este notebook escreve em `20_processed/`.*

### Path Setup (run first)
*Configuração do caminho (execute primeiro)*

In [ ]:
import os
import numpy as np
import pandas as pd

# Load the cleaned dataset from 2.2.3 | Carregar o conjunto de dados limpo do 2.2.3
DATA_CLEANED_DIR = '../../data/10_cleaned'
FILE_NAME = 'datania_households_clean.csv'
clean_path = os.path.join(DATA_CLEANED_DIR, FILE_NAME)

df = pd.read_csv(clean_path, dtype={'hh_id': str, 'region_code': str})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded:', df.shape)
df.head()

---

## Task 1: Recode: per-capita, bands, and label maps
*Tarefa 1: Recodificar: valores per capita, escalões e mapas de rótulos*

Recoding turns raw inputs into variables that are easier to analyse and report.

*A recodificação transforma os dados de origem em variáveis mais fáceis de analisar e de reportar.*

In [ ]:
# Income per household member (vectorised: column / column)
# Rendimento por membro do agregado (vetorizado: coluna / coluna)
df['income_per_capita'] = # your code here | o seu código aqui

df[['hh_id', 'income_dkw', 'hh_size', 'income_per_capita']].head()

In [ ]:
# Age bands with pd.cut: Child (0-18), Adult (18-65), Elderly (65-120)
# Escalões etários com pd.cut: Child (0-18), Adult (18-65), Elderly (65-120)
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 18, 65, 120],
    labels= # your code here | o seu código aqui
)
df['age_group'].value_counts(dropna=False)

In [ ]:
# Map education codes to labels | Mapear os códigos de educação para rótulos
education_map = {1: 'Primary', 2: 'Secondary', 3: 'Tertiary', 4: 'Higher levels'}
df['education_label'] = df['education_code']. # your code here | o seu código aqui

df['education_label'].value_counts(dropna=False)

**Questions:**

- `map()` returns `NaN` for any code not in the dictionary. Which households end up with a missing `education_label`, and why (think back to 2.2.3)?
- An income of `NaN` divided by `hh_size` gives what? Check `income_per_capita` for households whose income was missing.

***Perguntas:***

- *`map()` devolve `NaN` para qualquer código que não esteja no dicionário. Que agregados ficam sem `education_label` e porquê (recorde o 2.2.3)?*
- *Um rendimento `NaN` dividido por `hh_size` dá o quê? Verifique `income_per_capita` nos agregados cujo rendimento estava em falta.*

---

## Task 2: Conditional assignment with `np.where()` and `.loc[]`
*Tarefa 2: Atribuição condicional com `np.where()` e `.loc[]`*

`np.where()` is a vectorised IF: *if condition, value A, else value B*. `.loc[]` updates **existing** values that match a condition.

*`np.where()` é um SE vetorizado: *se a condição, valor A, senão valor B*. `.loc[]` atualiza valores **existentes** que correspondem a uma condição.*

In [ ]:
# Classify by population density (a different signal than the reported urban_rural)
# Classificar pela densidade populacional (um sinal diferente do urban_rural declarado)
df['area_type'] = np.where(df['pop_density'] > 500, # your code here: 'Urban', 'Rural' | o seu código aqui )

df[['hh_id', 'pop_density', 'urban_rural', 'area_type']].head(8)

In [ ]:
# Bucket household size by updating an existing column with .loc[]
# Agrupar a dimensão do agregado atualizando uma coluna existente com .loc[]
df['hh_category'] = 'Standard'
df.loc[df['hh_size'] >= 7, 'hh_category'] = # your code here: 'Large' | o seu código aqui: 'Large'
df.loc[df['hh_size'] <= 2, # your code here | o seu código aqui ] = 'Small'

df[['hh_id', 'hh_size', 'hh_category']].head(10)

**Questions:**

- Where the reported `urban_rural` and the density-based `area_type` disagree, which would you trust, and how would you investigate?
- `np.where()` treats a `NaN` population density as "not > 500" and labels it `Rural`. Is that the behaviour you want? How could you make the unknown explicit?

***Perguntas:***

- *Onde o `urban_rural` declarado e o `area_type` baseado na densidade divergem, em qual confiaria e como investigaria?*
- *`np.where()` trata uma densidade populacional `NaN` como "não > 500" e classifica-a como `Rural`. É esse o comportamento que quer? Como poderia tornar o desconhecido explícito?*

---

## Task 3: Clean column creation with `assign()`
*Tarefa 3: Criação de colunas de forma limpa com `assign()`*

`assign()` returns a **new** DataFrame, which keeps your code in a readable chain. With lambdas it can even build a column that depends on one created earlier in the same call.

*`assign()` devolve um **novo** DataFrame, o que mantém o seu código numa cadeia legível. Com lambdas pode até construir uma coluna que depende de outra criada na mesma chamada.*

In [ ]:
# Create two independent columns at once | Criar duas colunas independentes de uma só vez
df = df.assign(
    income_thousands = df['income_dkw'] / 1000,
    pop_density_log = # your code here: logarithmic value | o seu código aqui: valor logarítmico
)
df[['hh_id', 'income_dkw', 'income_thousands', 'pop_density', 'pop_density_log']].head()

In [ ]:
# Dependent columns: high_income is derived from income_per_capita created in the same call
# Colunas dependentes: high_income deriva de income_per_capita criada na mesma chamada
df = df.assign(
    income_per_capita = lambda x: x['income_dkw'] / x['hh_size'],
    high_income = lambda x: np.where( # your code here: x['income_per_capita'] > 25000, 'Yes', 'No' | o seu código aqui )
)
df[['hh_id', 'income_per_capita', 'high_income']].head()

**Question:** Why must `high_income` reference `lambda x: x[...]` instead of `df[...]`? What is `x` at that point in the chain?

***Pergunta:** Porque é que `high_income` tem de usar `lambda x: x[...]` em vez de `df[...]`? O que é `x` nesse ponto da cadeia?*

---

## Task 4: Multi-way categories with `np.select()`
*Tarefa 4: Categorias com várias saídas usando `np.select()`*

For more than two outcomes, `np.select()` is cleaner than nested `np.where()`. It evaluates conditions **in order** and takes the first match; `default` handles everything else (including `NaN`).

*Para mais de duas saídas, `np.select()` é mais claro do que `np.where()` encadeados. Avalia as condições **por ordem** e usa a primeira que corresponde; `default` trata de tudo o resto (incluindo `NaN`).*

In [ ]:
# Conditions are evaluated in order | As condições são avaliadas por ordem
conditions = [
    df['income_dkw'] < 40000,
    df['income_dkw'] < 70000,
    df['income_dkw'] >= 70000,
]
choices = ['Low', 'Medium', 'High']

df['income_band'] = np.select(conditions, choices, default= # your code here: 'Unknown' | o seu código aqui: 'Unknown' )
df['income_band'].value_counts()

**Questions:**

- How many households fall into `Unknown`? What do they have in common?
- Why does the **order** of the conditions matter? What would happen if `>= 70000` came first?

***Perguntas:***

- *Quantos agregados ficam em `Unknown`? O que têm em comum?*
- *Porque é que a **ordem** das condições importa? O que aconteceria se `>= 70000` viesse primeiro?*

---

## Task 5: Custom logic with `apply()`
*Tarefa 5: Lógica personalizada com `apply()`*

When built-in vectorised operations are not enough, `apply()` runs your own function on every value (or every row). Prefer vectorised code when you can (`apply()` is slower), but it is invaluable for complex logic.

*Quando as operações vetorizadas nativas não chegam, `apply()` executa a sua própria função em cada valor (ou em cada linha). Prefira código vetorizado sempre que possível (`apply()` é mais lento), mas é indispensável para lógica complexa.*

In [ ]:
# A named function for multi-step logic | Uma função com nome para lógica de vários passos
def classify_size(size):
    if size <= 2:
        return 'Small'
    elif size <= 5:
        return 'Medium'
    else:
        return 'Large'

df['hh_size_class'] = df['hh_size']. # your code here: apply(classify_size) | o seu código aqui
df['hh_size_class'].value_counts()

In [ ]:
# A lambda for simple one-line logic | Uma lambda para lógica simples de uma linha
df['high_income_flag'] = df['income_dkw'].apply(lambda x: # your code here: 'Yes' if x > 50000 else 'No' | o seu código aqui )
df[['hh_id', 'income_dkw', 'high_income_flag']].head()

In [ ]:
# A named function is clearer than a lambda once the logic has guards.
# apply(axis=1) passes one ROW at a time, so the function can read several columns.
# Uma função com nome é mais clara do que uma lambda quando a lógica tem verificações.
# apply(axis=1) passa uma LINHA de cada vez, para a função poder ler várias colunas.
def compute_income_per_capita(row):
    """Income per household member, or NaN if inputs are invalid.
    Rendimento por membro do agregado, ou NaN se os dados de entrada forem inválidos."""
    income, hh_size = row['income_dkw'], row['hh_size']
    if pd.isna(income) or pd.isna(hh_size) or hh_size <= 0:
        return np.nan
    return round(income / hh_size, 2)

df['per_capita_safe'] = df.apply( # your code here: compute_income_per_capita, axis=1 | o seu código aqui )
df[['hh_id', 'income_dkw', 'hh_size', 'per_capita_safe']].head()

**Reusable cleaning functions + `apply`**

When a *new* messy extract arrives, you can wrap the cleaning rules from 2.2 in a reusable function and `apply()` it: the natural home for `clean_income` and `standardise_date`. The pipeline data is already clean here, so we demonstrate on sample values.

***Funções de limpeza reutilizáveis + `apply`***

*Quando chega uma *nova* extração desorganizada, pode encapsular as regras de limpeza do 2.2 numa função reutilizável e aplicá-la com `apply()`: é o lugar natural para `clean_income` e `standardise_date`. Aqui os dados do fluxo já estão limpos, por isso demonstramos com valores de exemplo.*

In [ ]:
# Task 5: build the whole function yourself.
# clean_income: clean ONE raw income value and return a float (or NaN):
# handle NaN, strip " "/"Ar"/",", map text codes ("unknown", "NA", ...) to NaN, then convert to a number.
# Tarefa 5: construa a função completa por si.
# clean_income: limpar UM valor bruto de rendimento e devolver um float (ou NaN):
# tratar NaN, retirar " "/"Ar"/",", mapear códigos de texto ("unknown", "NA", ...) para NaN e converter para número.
def clean_income(value):
    # your code here: build the full function body
    # o seu código aqui: construa o corpo completo da função
    return  # your cleaned value, or np.nan | o seu valor limpo, ou np.nan

sample_income = pd.Series(["Ar 32,000", "45 000", "unknown", "1,200,000", np.nan])
sample_income.apply(clean_income)

In [ ]:
# Build the whole function yourself.
# standardise_date: turn ONE survey-date string into a datetime (or NaT):
# handle NaN / "not recorded"; accept MM/DD/YYYY and YYYY/MM/DD (use .split());
# fix an inverted month (the middle part > 12); then parse with pd.to_datetime.
# Construa a função completa por si.
# standardise_date: transformar UM texto de data de inquérito num datetime (ou NaT):
# tratar NaN / "not recorded"; aceitar MM/DD/AAAA e AAAA/MM/DD (use .split());
# corrigir um mês invertido (a parte do meio > 12); depois converter com pd.to_datetime.
def standardise_date(value):
    # your code here: build the full function body
    # o seu código aqui: construa o corpo completo da função
    return  # a datetime, or pd.NaT | um datetime, ou pd.NaT

sample_dates = pd.Series(["03/15/2025", "2025/01/18", "2025-13-01", "not recorded", "2025-01-10"])
sample_dates.apply(standardise_date)

**Questions:**

- When is a **named function** better than a **lambda**? When is the lambda fine?
- The lambda flag treats `NaN > 50000` as `False` -> `'No'`. Is silently labelling unknown income as "not high" safe? How would you guard against it?

***Perguntas:***

- *Quando é que uma **função com nome** é melhor do que uma **lambda**? Quando é que a lambda chega?*
- *A lambda trata `NaN > 50000` como `False` -> `'No'`. É seguro rotular silenciosamente um rendimento desconhecido como "não elevado"? Como se protegeria contra isso?*

---

## Task 6: Save the feature table to `20_processed/`
*Tarefa 6: Guardar a tabela de variáveis em `20_processed/`*

Cleaned data lives in `10_cleaned/`; derived/analysis tables go in `20_processed/`. Save the enriched DataFrame there for the merging step in 2.2.5.

*Os dados limpos ficam em `10_cleaned/`; as tabelas derivadas ou de análise vão para `20_processed/`. Guarde aí o DataFrame enriquecido para o passo de junção no 2.2.5.*

In [ ]:
DATA_PROC_DIR = '../../data/20_processed'
os.makedirs(DATA_PROC_DIR, exist_ok=True)
out_path = os.path.join(DATA_PROC_DIR, 'datania_households_features.csv')

df.to_csv( # your code here: index=False | o seu código aqui: index=False )
print('Saved:', out_path, '|', df.shape)

In [ ]:
# Reload to confirm it round-trips | Voltar a carregar para confirmar que está correto
check = pd.read_csv(out_path, dtype={'hh_id': str, 'region_code': str})
print('Reloaded:', check.shape)
check.head()

**Questions:**

- How many columns did you add compared with the cleaned input?
- Which of your new columns are **vectorised** (fast) and which used `apply()` (slower)? On a million-row file, which would you rewrite first?

***Perguntas:***

- *Quantas colunas acrescentou em relação aos dados limpos de entrada?*
- *Quais das suas novas colunas são **vetorizadas** (rápidas) e quais usaram `apply()` (mais lento)? Num ficheiro com um milhão de linhas, qual reescreveria primeiro?*